# Create unified output file

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../data'

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_PATH, 'resids.parquet')
print(bool(os.path.exists(FINAL_DOC)))

True


In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('._'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
files = [f for f in grab_all_files(OUTPUT_PATH, '.parquet') if (not f.startswith('._'))]

I found a ton of additional data that I missed in CANDOR. And so to make up for that, I reran everything for it. What's here is a sample, but it counts for a relatively large portion of data even on its own. I'm adding it in here so that we can have, hopefully, a single monstrously large dataset.

In [5]:
files += [f for f in grab_all_files(os.path.join(DATA_PATH, 'ckpts-candor'), '.parquet') if (not f.startswith('._'))]

In [6]:
print(files)

['../data/lme_ready/cha-ceda-analysis.parquet', '../data/lme_ready/null-ceda-analysis.parquet', '../data/lme_ready/grace-ceda-analysis.parquet', '../data/lme_ready/cha-coraal-ceda-analysis.parquet', '../data/lme_ready/cha-CABNC-ceda-analysis.parquet', '../data/lme_ready/cha-MICASE-ceda-analysis.parquet', '../data/ckpts-candor/28.parquet', '../data/ckpts-candor/9.parquet', '../data/ckpts-candor/25.parquet', '../data/ckpts-candor/31.parquet', '../data/ckpts-candor/32.parquet', '../data/ckpts-candor/5.parquet', '../data/ckpts-candor/26.parquet', '../data/ckpts-candor/24.parquet', '../data/ckpts-candor/33.parquet', '../data/ckpts-candor/30.parquet', '../data/ckpts-candor/4.parquet', '../data/ckpts-candor/6.parquet', '../data/ckpts-candor/27.parquet', '../data/ckpts-candor/8.parquet', '../data/ckpts-candor/3.parquet', '../data/ckpts-candor/17.parquet', '../data/ckpts-candor/29.parquet', '../data/ckpts-candor/7.parquet', '../data/ckpts-candor/0.parquet', '../data/ckpts-candor/1.parquet', '..

## Processing the data

Everything so far is saved into individual documents. This takes all of that and merges them into a single over-arching document, which is slightly less unwieldy.

In [7]:
start_from_turn = 5
min_acceptable_tokens = 5

In [8]:
df = []

In [9]:
for f in tqdm(files):
    print('\n', f)
    if f.endswith('.tsv'):
        df += [pd.read_table(f, sep='\t')]

        #doing this only once to decrease storage requirements
        os.remove(f)
        df[-1].to_parquet(f.replace('.tsv', '.parquet'))

    elif f.endswith('.parquet'):
        df += [pd.read_parquet(f)]

    else:
        df += [pd.read_csv(f, sep='\t')]

        #doing this only once to decrease storage requirements
        os.remove(f)
        df[-1].to_parquet(f.replace('.csv', '.parquet'))

    if ('/cha-' in f) or ('grace-' in f):
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]

    elif ('/candor-' in f) or ('ckpts-candor' in f):
        df[-1]['conversation_id'] = df[-1]['conversation_id']
        df[-1]['corpus'] = 'CANDOR'

    elif '/null-ceda' in f:
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]
        df[-1]['x_turn_id'] = [int(i.split('-')[-1]) for i in df[-1]['x_turn_id']]

    else:
        if 'conversation_id' in df[-1].columns:
            0
        else:
            df[-1]['conversation_id'] = f

    df[-1] = df[-1].loc[
        (df[-1]['nx'] >= min_acceptable_tokens)
        & (df[-1]['ny'] >= min_acceptable_tokens)
        & (df[-1]['x_turn_id'] >= start_from_turn)
    ]

    df[-1]['dyad'] = (df[-1]['x_speaker'] + '-' + df[-1]['y_speaker']).replace('\t', '-')
    df[-1]['turn_delta'] = (df[-1]['y_turn_id'] - df[-1]['x_turn_id'])

    df[-1] = df[-1][['Hxy', 'nx', 'ny', 'turn_delta', 'self',
                     'x_speaker', 'y_speaker',
                      'corpus', 'conversation_id', 'x_turn_id', 'y_turn_id']]

    df[-1]['x_turn_id'] = df[-1]['conversation_id'].astype(str) + '-' + df[-1]['x_turn_id'].astype(str)
    df[-1]['y_turn_id'] = df[-1]['conversation_id'].astype(str) + '-' + df[-1]['y_turn_id'].astype(str)

    df[-1]['null'] = bool('null-' in f)
df = pd.concat(df, ignore_index=True)

df.shape

  0%|          | 0/40 [00:00<?, ?it/s]


 ../data/lme_ready/cha-ceda-analysis.parquet


  0%|          | 0/8028150 [00:00<?, ?it/s]


 ../data/lme_ready/null-ceda-analysis.parquet


  0%|          | 0/4110515 [00:00<?, ?it/s]

...

  0%|          | 0/12723777 [00:00<?, ?it/s]


 ../data/lme_ready/cha-MICASE-ceda-analysis.parquet


  0%|          | 0/8262622 [00:00<?, ?it/s]


 ../data/ckpts-candor/28.parquet

 ../data/ckpts-candor/9.parquet

 ../data/ckpts-candor/25.parquet

 ../data/ckpts-candor/31.parquet

 ../data/ckpts-candor/32.parquet

 ../data/ckpts-candor/5.parquet

 ../data/ckpts-candor/26.parquet

 ../data/ckpts-candor/24.parquet

 ../data/ckpts-candor/33.parquet

 ../data/ckpts-candor/30.parquet

 ../data/ckpts-candor/4.parquet

 ../data/ckpts-candor/6.parquet

 ../data/ckpts-candor/27.parquet

 ../data/ckpts-candor/8.parquet

 ../data/ckpts-candor/3.parquet

 ../data/ckpts-candor/17.parquet

 ../data/ckpts-candor/29.parquet

 ../data/ckpts-candor/7.parquet

 ../data/ckpts-candor/0.parquet

 ../data/ckpts-candor/1.parquet

 ../data/ckpts-candor/2.parquet

 ../data/ckpts-candor/10.parquet

 ../data/ckpts-candor/11.parquet

 ../data/ckpts-candor/12.parquet

 ../data/ckpts-candor/13.parquet

 ../data/ckpts-candor/14.parquet

 ../data/ckpts-candor/15.parquet

 ../data/ckpts-candor/16.parquet

 ../data/ckpts-candor/18.parquet

 ../data/ckpts-candor/1

(44007049, 12)

In [10]:
df.isna().sum()

Hxy                0
nx                 0
ny                 0
turn_delta         0
self               0
x_speaker          0
y_speaker          0
corpus             0
conversation_id    0
x_turn_id          0
y_turn_id          0
null               0
dtype: int64

#### Checking on and dealing with null-distribution test results

In [11]:
df['null'].value_counts()

null
False    41815356
True      2191693
Name: count, dtype: int64

#### other checks

In [13]:
df['conversation_id'].nunique()

4284

In [14]:
df.isna().sum()

Hxy                0
nx                 0
ny                 0
turn_delta         0
self               0
x_speaker          0
dyad               0
corpus             0
conversation_id    0
x_turn_id          0
null               0
dtype: int64

In [15]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False


## Relabel/add a column for per comparison sample $\Delta$

In my initial work, it didn't really matter what number sample each value was. So it didn't matter if $t \pm 1$ was a self or other comment. Who cared. But for merging the data and for calculating allan variance, this is actually kinda important. So now I'm re-adding this in there.

This all might be moot. There is very much a world in which what I actually need is a much simpler comparison between turn $\Delta$ values. Perhaps using Kolomogorov-Smirnov testing between each sample is sufficient? All I need, after all, is to prove that power-scaling is occurring here. And Allan-variance is just one test (though a well-trodden one) that can be used to assess that.

In [12]:
# df['groups'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)
df['groups'] = df['x_turn_id'].astype(str)

In [13]:
df['sample_delta'] = df.groupby(by=['groups', 'self']).cumcount() + 1

In [14]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,y_speaker,corpus,conversation_id,x_turn_id,y_turn_id,null,groups,sample_delta
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-7,False,GCSAusE-GCSAusE-01-6,1
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-8,False,GCSAusE-GCSAusE-01-6,2
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-10,False,GCSAusE-GCSAusE-01-6,3
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-12,False,GCSAusE-GCSAusE-01-6,4
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-13,False,GCSAusE-GCSAusE-01-6,5


## Tidying the data

Making sure there aren't any erroneous `\t` elements in there.

In [19]:
# cleaning
for col in list(df):
    print(col)
    df[col] = [it.replace('\t', '') if isinstance(it,str) else it for it in tqdm(df[col])]

Hxy


  0%|          | 0/44007049 [00:00<?, ?it/s]

nx


  0%|          | 0/44007049 [00:00<?, ?it/s]

ny


...

  0%|          | 0/44007049 [00:00<?, ?it/s]

groups


  0%|          | 0/44007049 [00:00<?, ?it/s]

sample_delta


  0%|          | 0/44007049 [00:00<?, ?it/s]

## Save outputs

Because that's the whole f'ing point.

In [20]:
# df.to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')

In [15]:
df.to_parquet(FINAL_DOC, engine='fastparquet', compression='snappy')

#### Check that data loads after saving

In [22]:
df_ = pd.read_parquet(FINAL_DOC)

In [ ]:
del df_